In [2]:
import torch
import torch.nn as nn
import math

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        assert d_model % num_heads == 0, "d_model must divisible by num of heads"
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = nn.Dropout(dropout)
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q: (batch, num_heads, seq_len_q, d_k)
        K: (batch, num_heads, seq_len_k, d_k)
        V: (batch, num_heads, seq_len_k, d_v)
        
        Return: 
            weights: (batch, num_heads, seq_len_q, seq_len_k)
            attention: (batch, num_heads, seq_len_q, d_v)
        """
        d_k = Q.size(-1)
        # weights dimension: (batch, num_heads, seq_len_q, seq_len_k)
        scores = torch.matmul(Q, K.transpose(-1, -2))
        scores = scores/math.sqrt(d_k)
        if mask != None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        attention = torch.matmul(weights, V)
        return attention, weights

    def forward(self, x, mask=None):
        # x dimension: (batch, seq_len, d_model)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # (batch, seq_len, d_model) -> (batch, seq_len, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        batch = Q.size(0)
        Q = Q.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, -1, self.num_heads, self.d_k).transpose(1, 2)

        # attention: (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v) 
        #     -> (batch, seq_len, d_model)
        attention, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        attention = attention.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        output = self.W_O(attention)
        return output

class ForwardFeedNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.relu = nn.ReLU()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        result = self.linear2(
            self.dropout(
                self.relu(self.linear1(x))
            )
        )
        return result

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000)) / d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = ForwardFeedNetwork(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x = x + self.dropout1(self.attention(self.norm1(x), mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len=512, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.token_embedding.weight)

    def forward(self, input_ids, mask=None):
        # input_ids dimension: (batch, seq_len) -> x dimension (batch, seq_len, d_model)
        x = self.token_embedding(input_ids)
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x, mask)

        pooled = x.mean(dim=1)
        normed = self.final_norm(pooled)
        logits = self.classifier(normed)
        return logits


model = TransformerEncoder(10000, 128, 4, 512, 2, 2, max_len=256, dropout=0.1)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

dummy_input = torch.randint(0, 1000, (4, 32))
output = model(dummy_input)    
print(f"Output shape: {output.shape}")
        
        

Number of parameters: 1,677,058
Output shape: torch.Size([4, 2])


In [21]:
def test_shapes():
    batch, seq_len, d_model, num_heads = 2, 10, 128, 4

    mha = MultiHeadAttention(d_model, num_heads)
    x = torch.randn(batch, seq_len, d_model)
    output = mha(x)
    assert output.shape == (batch, seq_len, d_model), f"MultiHeadAttention shape wrong: {output.shape}"
    print(f"MultiHeadAttention shape correct")

    ffn = ForwardFeedNetwork(d_model, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    output = ffn(x)
    assert output.shape == (batch, seq_len, d_model), f"ForwardFeedNetwork shape wrong: {output.shape}"
    print(f"ForwardFeedNetwork shape correct")

    # d_model, num_heads, d_ff, dropout=0.1
    encoder_layer = EncoderLayer(d_model, num_heads, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    output = encoder_layer(x)
    assert output.shape == (batch, seq_len, d_model), f"Encoder Layer shape wrong: {output.shape}"
    print(f"Encoder Layer shape correct")

    # vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len=512, dropout=0.1):
    model = TransformerEncoder(10000, d_model, num_heads, 4*d_model, 2, 2, max_len=512, dropout=0.1)
    input_ids = torch.randint(0, 10000, (batch, seq_len))
    output = model(input_ids)
    assert output.shape == (batch, 2), f"Transformer Encoder shape wrong: {output.shape}"
    print(f"Transformer Encoder shape correct")
    
test_shapes()

MultiHeadAttention shape correct
ForwardFeedNetwork shape correct
Encoder Layer shape correct
Transformer Encoder shape correct


In [24]:
def test_gradient():
    model = TransformerEncoder(10000, 128, 4, 512, 2, 2, max_len=512, dropout=0.1)
    x = torch.randint(0, 10000, (2, 10))
    targets = torch.tensor([0, 1])

    logits = model(x)
    loss = nn.CrossEntropyLoss()(logits, targets)
    loss.backward()

    for name, parameter in model.named_parameters():
        print(f"{name}")
        if parameter.requires_grad:
            assert parameter.grad is not None, f"No gradient for {name}"
            assert parameter.grad.abs().sum() > 0, f"Zero gradient for {name}"

    print(f"All parameters have non-zero gradients")

test_gradient()
        

token_embedding.weight
layers.0.attention.W_Q.weight
layers.0.attention.W_Q.bias
layers.0.attention.W_K.weight
layers.0.attention.W_K.bias
layers.0.attention.W_V.weight
layers.0.attention.W_V.bias
layers.0.attention.W_O.weight
layers.0.attention.W_O.bias
layers.0.ffn.linear1.weight
layers.0.ffn.linear1.bias
layers.0.ffn.linear2.weight
layers.0.ffn.linear2.bias
layers.0.norm1.weight
layers.0.norm1.bias
layers.0.norm2.weight
layers.0.norm2.bias
layers.1.attention.W_Q.weight
layers.1.attention.W_Q.bias
layers.1.attention.W_K.weight
layers.1.attention.W_K.bias
layers.1.attention.W_V.weight
layers.1.attention.W_V.bias
layers.1.attention.W_O.weight
layers.1.attention.W_O.bias
layers.1.ffn.linear1.weight
layers.1.ffn.linear1.bias
layers.1.ffn.linear2.weight
layers.1.ffn.linear2.bias
layers.1.norm1.weight
layers.1.norm1.bias
layers.1.norm2.weight
layers.1.norm2.bias
final_norm.weight
final_norm.bias
classifier.weight
classifier.bias
All parameters have non-zero gradients


In [37]:
# vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len=512, dropout=0.1)
def test_overfit():
    model = TransformerEncoder(100, 64, 4, 256, 2, 2)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    x = torch.randint(0, 100, (8, 16))
    y = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1])

    model.train()
    for epoch in range(100):
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            acc = (logits.argmax(dim=1) == y).float().mean()
            print(f"Epoch {epoch + 1}: "
                  f"loss = {loss.item():.4f}, "
                  f"acc = {acc.item():.2%}")

    final_acc = (model(x).argmax(dim=1) == y).float().mean()
    assert final_acc > 0.9, f"Failed to overfit! Final Accuracy {final_acc:.2%}"
    print(f"Overfit success! Accuracy {final_acc:.2%}")
    
test_overfit()      

    

Epoch 20: loss = 0.6774, acc = 50.00%
Epoch 40: loss = 0.2149, acc = 100.00%
Epoch 60: loss = 0.0397, acc = 100.00%
Epoch 80: loss = 0.0159, acc = 100.00%
Epoch 100: loss = 0.0020, acc = 100.00%
Overfit success! Accuracy 100.00%
